# 03 - Model Implementation & Comparison

Covers capstone Step 4: five candidate generators compared under the CI-aware
leave-last-order-out protocol, three tuned rankers on the temporal-holdout task
with hard-negative sampling, and the end-to-end two-stage evaluation that
decides the final pipeline. Every run is logged to MLflow; artifacts and
metrics land in `models/`.

**Two artifact builds, per the evaluation plan:** the candidate-generator
comparison uses leave-last-order-out artifacts (usable range minus each repeat
buyer's held-out last order); ranker features, hard negatives, and the
end-to-end holdout evaluation use feature-window-only artifacts.

In [1]:
import json
import sys
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import seaborn as sns
import yaml

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from src.data import load_table
from src.evaluate import (
    bootstrap_ci, coverage_and_longtail, evaluate_model, leave_last_order_split,
    reachability,
)
from src.features import (
    assert_max_ts, cold_fill_stats, customer_features, customer_geo, interactions,
    load_centroids, load_clean_items, load_clean_orders, load_clean_products,
    load_clean_reviews, load_windows, pair_features, product_features,
    reviews_before, sample_negatives, seller_features,
)
from src.models.candidate_gen import (
    ContentBased, HybridRecommender, InteractionMatrix, ItemItemCF,
    PopularityRecommender, SVDRecommender, top_k_from_scores,
)
from src.recommend import Router, regional_popularity

sns.set_theme(style="whitegrid")
SEED = 42
FIG_DIR = REPO_ROOT / "reports" / "figures"
MODELS_DIR = REPO_ROOT / "models"
CFG = yaml.safe_load((REPO_ROOT / "configs" / "models.yaml").read_text())

mlflow.set_tracking_uri(f"sqlite:///{(REPO_ROOT / 'mlruns.db').as_posix()}")
mlflow.set_experiment("capstone")

windows = load_windows()
FW, LW, HO = windows["feature_window"], windows["label_window"], windows["holdout"]
USABLE = (FW[0], HO[1])

delivered, _ = load_clean_orders()
items, _ = load_clean_items()
reviews, _ = load_clean_reviews()
products, _ = load_clean_products()
payments = load_table("order_payments")
centroids = load_centroids()
geo = customer_geo()
geo_region = geo["region"].to_dict()

# leakage audit (purchases + review creations, both artifact builds)
fw_inter = interactions(delivered, items, *FW)
assert_max_ts(fw_inter, "order_purchase_timestamp", FW[1])
assert_max_ts(
    reviews_before(reviews, FW[1]).merge(fw_inter[["order_id"]].drop_duplicates(), on="order_id"),
    "review_creation_date", FW[1],
)
print(f"leakage audit passed; usable range {USABLE[0].date()} -> {USABLE[1].date()}")

C:\Users\jmonz\Documents\GitHub\ecommerce-recommender-capstone\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


leakage audit passed; usable range 2017-01-01 -> 2018-09-01


## Candidate generation - leave-last-order-out protocol

Repeat buyers' last orders are held out; Stage-1 artifacts fit on everything
else in the usable range. 30% of eval users tune the SVD rank and hybrid
weight; the untouched 70% produce the comparison table. Hit rates over a
33k-product catalog are tiny, so every headline number carries a bootstrap
CI and winners are only claimed when intervals separate.

In [2]:
train_pairs, held = leave_last_order_split(delivered, items, *USABLE)
im = InteractionMatrix(train_pairs, products["product_id"].to_numpy())

held_idx = held.assign(item=im.item_index.get_indexer(held["product_id"]))
eval_sets = held_idx.groupby("customer_unique_id")["item"].agg(set)

rng = np.random.default_rng(SEED)
users = eval_sets.index.to_numpy()
val_mask = rng.random(len(users)) < CFG["candidate_generation"]["val_share"]
val_users, test_users = eval_sets[val_mask], eval_sets[~val_mask]

print(f"repeat buyers evaluated: {len(eval_sets)} (val {len(val_users)}, test {len(test_users)})")
print(f"training interactions: {len(train_pairs)}")
print(f"reachability ceiling (held-out products seen in training): {reachability(held, train_pairs):.1%}")
print(f"held-out products per user: {held_idx.groupby('customer_unique_id').size().mean():.2f} mean")

repeat buyers evaluated: 2789 (val 840, test 1949)
training interactions: 96883
reachability ceiling (held-out products seen in training): 77.6%
held-out products per user: 1.07 mean


In [3]:
pop = PopularityRecommender().fit(im)
cf = ItemItemCF().fit(im)
content = ContentBased().fit(im, products.set_index("product_id"))
print(f"item-item similarity: {cf.S.nnz} nonzero pairs "
      f"({cf.S.nnz / (cf.S.shape[0] ** 2):.2e} dense share)")

# SVD rank sweep on validation users
svd_curve = {}
for k in CFG["candidate_generation"]["svd_components"]:
    svd_k = SVDRecommender(n_components=k).fit(im)
    res = evaluate_model(f"svd{k}", lambda c, ui, m=svd_k: m.scores(im, ui, customer_id=c),
                         im, val_users)
    svd_curve[k] = res["hit@10"].mean()
    print(f"SVD k={k:<4} val HitRate@10 {svd_curve[k]:.4f}")
K_SVD = max(svd_curve, key=svd_curve.get)
svd = SVDRecommender(n_components=K_SVD).fit(im)

# hybrid weight sweep on validation users
w_curve = {}
for w in CFG["candidate_generation"]["hybrid_weights"]:
    hy_w = HybridRecommender(cf, content, w=w)
    res = evaluate_model(f"hybrid{w}", lambda c, ui, m=hy_w: m.scores(im, ui), im, val_users)
    w_curve[w] = res["hit@10"].mean()
    print(f"hybrid w={w:<5} val HitRate@10 {w_curve[w]:.4f}")
W_HYBRID = max(w_curve, key=w_curve.get)
hybrid = HybridRecommender(cf, content, w=W_HYBRID)
print(f"\nselected: SVD k={K_SVD}, hybrid w={W_HYBRID}")
print(f"note: hybrid val spread is {max(w_curve.values()) - min(w_curve.values()):.4f} "
      f"on 840 users - within noise, so w is a coarse choice, not a tuned claim")

item-item similarity: 8710 nonzero pairs (8.02e-06 dense share)


SVD k=16   val HitRate@10 0.0262


SVD k=32   val HitRate@10 0.0262


SVD k=64   val HitRate@10 0.0310


SVD k=128  val HitRate@10 0.0381


SVD k=256  val HitRate@10 0.0500


hybrid w=0.25  val HitRate@10 0.2167


hybrid w=0.5   val HitRate@10 0.2143


hybrid w=0.75  val HitRate@10 0.2131

selected: SVD k=256, hybrid w=0.25
note: hybrid val spread is 0.0036 on 840 users - within noise, so w is a coarse choice, not a tuned claim


In [4]:
MODELS = {
    "popularity": lambda c, ui: pop.scores(im, ui),
    "item_item_cf": lambda c, ui: cf.scores(im, ui),
    "content_based": lambda c, ui: content.scores(im, ui),
    f"svd_k{K_SVD}": lambda c, ui: svd.scores(im, ui, customer_id=c),
    f"hybrid_w{W_HYBRID}": lambda c, ui: hybrid.scores(im, ui),
}
rows, res_by_name = [], {}
for name, fn in MODELS.items():
    res = evaluate_model(name, fn, im, test_users)
    res_by_name[name] = res
    lo, hi = bootstrap_ci(res["hit@10"].to_numpy())
    cov = coverage_and_longtail(res.attrs["top10s"], pop.scores_, len(im.product_ids))
    row = {
        "model": name,
        "hit@10": res["hit@10"].mean(), "hit@10_lo": lo, "hit@10_hi": hi,
        "hit@50": res["hit@50"].mean(), "hit@100": res["hit@100"].mean(),
        "recall@10": res["recall@10"].mean(), "mrr": res["mrr"].mean(),
        "ndcg@10": res["ndcg@10"].mean(),
        "repurchase_share_of_hits": (res["repurchase_hit"].sum() / max(res["hit@10"].sum(), 1)),
        **cov,
    }
    rows.append(row)
    with mlflow.start_run(run_name=f"candidate_{name}"):
        mlflow.log_params({"stage": "candidate_generation", "model": name,
                           "protocol": "leave_last_order_out"})
        mlflow.log_metrics({k.replace("@", "_at_"): v for k, v in row.items()
                            if isinstance(v, float)})

cand_table = pd.DataFrame(rows).set_index("model")
print(cand_table.round(4).to_string())

# paired per-user difference (same eval users for every model) - stronger than
# comparing overlap of two independent CIs
diff = (res_by_name[f"hybrid_w{W_HYBRID}"]["hit@10"].to_numpy()
        - res_by_name["content_based"]["hit@10"].to_numpy())
lo_d, hi_d = bootstrap_ci(diff)
print(f"\npaired diff hybrid - content, hit@10: {diff.mean():+.4f} [{lo_d:+.4f}, {hi_d:+.4f}]")

               hit@10  hit@10_lo  hit@10_hi  hit@50  hit@100  recall@10     mrr  ndcg@10  repurchase_share_of_hits  coverage  longtail_share
model                                                                                                                                       
popularity     0.0308     0.0231     0.0385  0.0806   0.1216     0.0294  0.0129   0.0138                    0.2333    0.0003          0.0000
item_item_cf   0.0554     0.0452     0.0652  0.0564   0.0575     0.0545  0.0369   0.0410                    0.2593    0.0419          0.8331
content_based  0.1709     0.1539     0.1873  0.2088   0.2314     0.1679  0.1352   0.1409                    0.6907    0.3766          0.7921
svd_k256       0.0544     0.0446     0.0646  0.0754   0.0995     0.0539  0.0457   0.0465                    0.7830    0.0194          0.0251
hybrid_w0.25   0.2027     0.1837     0.2211  0.2411   0.2601     0.1994  0.1376   0.1506                    0.6177    0.3706          0.7604

paired diff 

In [5]:
# Category-level HitRate@10: does the top-10 contain the held-out CATEGORY?
# (workable base rate on a 71-category space vs 33k products)
prod_cat = products.set_index("product_id")["category"]
cat_of_idx = prod_cat.reindex(pd.Index(im.product_ids)).to_numpy()

cat_rows = {}
for name, fn in MODELS.items():
    hits = []
    for cust, held_i in test_users.items():
        held_cats = {cat_of_idx[i] for i in held_i if i >= 0}
        top = top_k_from_scores(fn(cust, im.user_items(cust)), 10)
        hits.append(float(bool(held_cats & set(cat_of_idx[top]))))
    cat_rows[name] = np.mean(hits)
print(pd.Series(cat_rows, name="category_hit@10").round(4).to_string())

popularity       0.4767
item_item_cf     0.5521
content_based    0.4767
svd_k256         0.4541
hybrid_w0.25     0.5018


## Ranker - hard-negative protocol

Positives: delivered (customer, product) purchases in the label window (train)
and holdout window (evaluation). Negatives per positive: 4 total - half drawn
from the customer's own Stage-1 candidate list (what the pipeline would
actually show; regional popularity for cold customers), half uniform random.
All features are feature-window only; tuning uses GroupKFold by customer.
The AUC/PR-AUC/F1 numbers compare rankers against each other - they are
artifacts of the sampling ratio, not absolute performance claims.

In [6]:
# feature-window artifacts (ranker + serving)
im_fw = InteractionMatrix(
    fw_inter[["customer_unique_id", "product_id"]].drop_duplicates(),
    products["product_id"].to_numpy(),
)
cf_fw = ItemItemCF().fit(im_fw)
content_fw = ContentBased().fit(im_fw, products.set_index("product_id"))
hybrid_fw = HybridRecommender(cf_fw, content_fw, w=W_HYBRID)
pop_fw = PopularityRecommender().fit(im_fw)
N_CAND = CFG["candidate_generation"]["n_candidates"]
global_top = top_k_from_scores(pop_fw.scores_, N_CAND)
regional_top = regional_popularity(fw_inter, im_fw.item_index, k=N_CAND)
router = Router(im_fw, hybrid_fw, global_top, regional_top,
                n_candidates=CFG["candidate_generation"]["n_candidates"])

cust_fw = customer_features(delivered, items, payments, reviews, products, FW)
prod_fw = product_features(delivered, items, reviews, products, FW)
sell_fw = seller_features(delivered, items, reviews, FW)
fills = cold_fill_stats(delivered, items, reviews, centroids, FW)
print("feature-window artefacts ready")

feature-window artefacts ready


In [7]:
def build_ranker_pairs(window, tag):
    """Positives + mixed hard/random negatives for one label window."""
    rng = np.random.default_rng(CFG["negative_sampling"]["seed"])
    pos = interactions(delivered, items, *window)[
        ["customer_unique_id", "product_id"]].drop_duplicates()
    pos_sets = pos.groupby("customer_unique_id")["product_id"].agg(set)

    hard_rows = []
    n_hard = int(CFG["negative_sampling"]["ratio"] * CFG["negative_sampling"]["hard_share"])
    for cust, bought in pos_sets.items():
        cands, _ = router.recommend(cust, region=geo_region.get(cust), k=None)
        cand_ids = im_fw.product_ids[cands]
        pool = [p for p in cand_ids if p not in bought]
        take = rng.choice(len(pool), size=min(n_hard * len(bought), len(pool)),
                          replace=False) if pool else []
        hard_rows += [(cust, pool[i]) for i in take]
    hard = pd.DataFrame(hard_rows, columns=["customer_unique_id", "product_id"])

    n_rand_ratio = CFG["negative_sampling"]["ratio"] - n_hard
    rand = sample_negatives(pos, products["product_id"].to_numpy(),
                            ratio=n_rand_ratio, seed=CFG["negative_sampling"]["seed"])
    pairs = pd.concat([pos.assign(y=1), hard.assign(y=0), rand.assign(y=0)],
                      ignore_index=True).drop_duplicates(
        ["customer_unique_id", "product_id"], keep="first")
    print(f"{tag}: {int((pairs.y == 1).sum())} pos, {int((pairs.y == 0).sum())} neg "
          f"({len(hard)} hard, {len(rand)} random before dedupe)")
    return pairs

train_pairs_r = build_ranker_pairs(LW, "train (label window)")
hold_pairs_r = build_ranker_pairs(HO, "holdout")

train (label window): 21260 pos, 85033 neg (42520 hard, 42517 random before dedupe)


holdout: 19299 pos, 77192 neg (38598 hard, 38596 random before dedupe)


In [8]:
def ranker_matrix(pairs):
    X_all = pair_features(pairs[["customer_unique_id", "product_id"]],
                          cust_fw, geo, prod_fw, sell_fw, centroids, fills)
    user_items_map = {c: im_fw.user_items(c) for c in pairs["customer_unique_id"].unique()}
    X_all["cf_signal"] = cf_fw.pair_scores(user_items_map, pairs, im_fw.item_index)
    X_all["has_cf_signal"] = (X_all["cf_signal"] > 0).astype(float)
    cat_share = (fw_inter.merge(products[["product_id", "category"]], on="product_id")
                 ["category"].value_counts(normalize=True))
    num = X_all.drop(columns=["customer_unique_id", "product_id", "p_category", "c_region"])
    num["p_category_freq"] = X_all["p_category"].map(cat_share).fillna(0.0)
    num = pd.concat([num, pd.get_dummies(X_all["c_region"], prefix="region", dtype=float)],
                    axis=1)
    return num

X_tr = ranker_matrix(train_pairs_r)
X_ho = ranker_matrix(hold_pairs_r)
X_ho = X_ho.reindex(columns=X_tr.columns, fill_value=0.0)
y_tr, y_ho = train_pairs_r["y"].to_numpy(), hold_pairs_r["y"].to_numpy()
g_tr = train_pairs_r["customer_unique_id"].to_numpy()
print(f"train {X_tr.shape}, holdout {X_ho.shape}")

# CF-signal sanity check: training positives vs holdout serving pairs. If this
# feature were a leakage machine it would look strong in training and be ~0 at
# serving time - the plot documents the actual gap.
fig, ax = plt.subplots(figsize=(7, 4))
tr_pos = X_tr.loc[y_tr == 1, "cf_signal"]
ho_all = X_ho["cf_signal"]
ax.hist(np.log1p(tr_pos), bins=50, alpha=0.6, density=True, label="train positives")
ax.hist(np.log1p(ho_all), bins=50, alpha=0.6, density=True, label="holdout serving pairs")
ax.set_xlabel("log1p(cf_signal)")
ax.set_title("CF-signal distribution: train positives vs serving pairs")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "03_cf_signal_sanity.png", dpi=150)
plt.close(fig)
print(f"cf_signal > 0: train positives {(tr_pos > 0).mean():.1%}, "
      f"holdout pairs {(ho_all > 0).mean():.1%}")

train (106293, 32), holdout (96491, 32)


cf_signal > 0: train positives 0.1%, holdout pairs 0.0%


In [9]:
# Feature-selection re-check on the real protocol (screen from notebook 02)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, f1_score, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

lr_l1 = make_pipeline(StandardScaler(),
                      LogisticRegression(l1_ratio=1, solver="liblinear", C=0.01,
                                         max_iter=2000, random_state=SEED))
lr_l1.fit(X_tr, y_tr)
coefs = lr_l1.named_steps["logisticregression"].coef_[0]
surv = pd.Series(coefs, index=X_tr.columns)
surv = surv[surv != 0].sort_values(key=np.abs, ascending=False)
print(f"L1 survivors at C=0.01 on hard-negative protocol ({len(surv)}/{X_tr.shape[1]}):")
print(surv.round(3).to_string())

L1 survivors at C=0.01 on hard-negative protocol (19/32):
p_popularity                   -0.849
p_has_sales                    -0.197
same_state                      0.065
p_category_satisfaction_rate   -0.054
category_match                 -0.052
p_category_freq                 0.049
p_product_weight_g             -0.049
p_product_description_lenght    0.046
p_product_photos_qty            0.043
p_median_price_w               -0.037
has_history                    -0.024
s_seller_review_mean            0.022
p_freight_ratio                -0.020
p_price_band                    0.018
has_cf_signal                   0.013
region_North                   -0.011
distance_km                     0.010
price_delta                    -0.008
c_median_item_price            -0.002


In [10]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (GridSearchCV, GroupKFold, RandomizedSearchCV,
                                     cross_val_predict)
from xgboost import XGBClassifier

gkf = GroupKFold(n_splits=CFG["cv_folds"])
R = CFG["rankers"]

searches = {
    "logistic_regression": GridSearchCV(
        make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, random_state=SEED)),
        {"logisticregression__C": R["logistic_regression"]["C"],
         "logisticregression__class_weight": R["logistic_regression"]["class_weight"]},
        scoring="roc_auc", cv=gkf, n_jobs=-1),
    "random_forest": GridSearchCV(
        RandomForestClassifier(random_state=SEED, n_jobs=-1),
        {k: v for k, v in R["random_forest"].items()},
        scoring="roc_auc", cv=gkf, n_jobs=1),
    "xgboost": RandomizedSearchCV(
        XGBClassifier(tree_method="hist", random_state=SEED, n_jobs=-1, eval_metric="auc"),
        R["xgboost"]["param_distributions"], n_iter=R["xgboost"]["n_iter"],
        scoring="roc_auc", cv=gkf, random_state=SEED, n_jobs=1),
}

rankers, ranker_rows = {}, []
for name, search in searches.items():
    search.fit(X_tr, y_tr, groups=g_tr)
    best = search.best_estimator_
    rankers[name] = best
    proba_ho = best.predict_proba(X_ho)[:, 1]
    proba_tr = best.predict_proba(X_tr)[:, 1]
    # tau from OUT-OF-FOLD predictions: training-fit probabilities are
    # optimistically separated for the tree models (RF train AUC ~0.95), which
    # would bias their thresholds; holdout still never picks its own tau
    proba_oof = cross_val_predict(best, X_tr, y_tr, cv=gkf, groups=g_tr,
                                  method="predict_proba", n_jobs=1)[:, 1]
    taus = np.quantile(proba_oof, np.linspace(0.5, 0.95, 19))
    tau = taus[np.argmax([f1_score(y_tr, proba_oof >= t) for t in taus])]
    row = {
        "model": name,
        "cv_auc": search.best_score_,
        "holdout_auc": roc_auc_score(y_ho, proba_ho),
        "holdout_pr_auc": average_precision_score(y_ho, proba_ho),
        "holdout_f1_at_tau": f1_score(y_ho, proba_ho >= tau),
        "tau": float(tau),
        "train_auc": roc_auc_score(y_tr, proba_tr),
    }
    ranker_rows.append(row)
    with mlflow.start_run(run_name=f"ranker_{name}"):
        mlflow.log_params({"stage": "ranker", "model": name, **{
            str(k): str(v) for k, v in search.best_params_.items()}})
        mlflow.log_metrics({k: float(v) for k, v in row.items() if k != "model"})
    print(f"{name}: best {search.best_params_}")

ranker_table = pd.DataFrame(ranker_rows).set_index("model")
print(ranker_table.round(4).to_string())

logistic_regression: best {'logisticregression__C': 10.0, 'logisticregression__class_weight': None}


random_forest: best {'max_depth': None, 'min_samples_leaf': 5, 'n_estimators': 300}


xgboost: best {'subsample': 1.0, 'scale_pos_weight': 1, 'n_estimators': 400, 'max_depth': 6, 'learning_rate': 0.2, 'colsample_bytree': 0.7}
                     cv_auc  holdout_auc  holdout_pr_auc  holdout_f1_at_tau     tau  train_auc
model                                                                                         
logistic_regression  0.6972       0.7847          0.4137             0.4750  0.2160     0.6977
random_forest        0.8519       0.8654          0.6189             0.5688  0.2819     0.9449
xgboost              0.8530       0.8641          0.6136             0.5783  0.2720     0.9248


## End-to-end two-stage evaluation (the decision table)

For every customer with a holdout purchase: route (warm → hybrid top-50, cold →
regional/global popularity), re-rank the candidate list with each ranker, take
the top 10, and score against what they actually bought. This is the only table
where candidate generators and rankers meet on one metric - it decides the
final pipeline and is the headline number for both decks.

In [11]:
ho_pos = interactions(delivered, items, *HO)[
    ["customer_unique_id", "product_id"]].drop_duplicates()
ho_sets_ids = ho_pos.groupby("customer_unique_id")["product_id"].agg(set)
print(f"holdout customers: {len(ho_sets_ids)}, purchases: {len(ho_pos)}")

cand_rows, route_of = [], {}
for cust in ho_sets_ids.index:
    cands, route = router.recommend(cust, region=geo_region.get(cust), k=None)
    route_of[cust] = route
    cand_rows.append(pd.DataFrame({
        "customer_unique_id": cust,
        "product_id": im_fw.product_ids[cands],
        "stage1_rank": np.arange(len(cands)),
    }))
cands_df = pd.concat(cand_rows, ignore_index=True)
routes = pd.Series(route_of)
print("routing shares:")
print(routes.value_counts(normalize=True).round(4).to_string())

X_e2e = ranker_matrix(cands_df).reindex(columns=X_tr.columns, fill_value=0.0)
print(f"end-to-end scoring matrix: {X_e2e.shape}")

holdout customers: 18390, purchases: 19299

routing shares:
regional_popularity    0.9849
hybrid_only            0.0151


end-to-end scoring matrix: (919500, 32)


In [12]:
e2e_hits = {}

def e2e_metrics(ordered_topk_by_cust, tag):
    hits, ndcgs = [], []
    from src.evaluate import rank_metrics
    for cust, held_ids in ho_sets_ids.items():
        top = ordered_topk_by_cust.get(cust)
        if top is None:
            hits.append(0.0); ndcgs.append(0.0); continue
        m = rank_metrics(np.asarray(top), held_ids, k_levels=(10,), ndcg_k=10)
        hits.append(m["hit@10"]); ndcgs.append(m["ndcg@10"])
    hits = np.array(hits)
    e2e_hits[tag] = hits
    lo, hi = bootstrap_ci(hits)
    warm = routes.reindex(ho_sets_ids.index).isin(["two_stage", "hybrid_only"]).to_numpy()
    wlo, whi = bootstrap_ci(hits[warm]) if warm.any() else (np.nan, np.nan)
    return {"pipeline": tag, "hit@10": hits.mean(), "hit@10_lo": lo, "hit@10_hi": hi,
            "ndcg@10": float(np.mean(ndcgs)),
            "hit@10_warm": hits[warm].mean() if warm.any() else np.nan,
            "warm_lo": wlo, "warm_hi": whi, "n_warm": int(warm.sum())}

results = []

# Stage-1 only: first 10 candidates in stage-1 order
stage1_top = {c: g.sort_values("stage1_rank")["product_id"].to_numpy()[:10]
              for c, g in cands_df.groupby("customer_unique_id")}
results.append(e2e_metrics(stage1_top, "stage1_only (routed hybrid/popularity)"))

# Global popularity for everyone (the floor)
pop_ids = im_fw.product_ids[global_top[:10]]
results.append(e2e_metrics({c: pop_ids for c in ho_sets_ids.index}, "popularity_only"))

# Stage-1 + each ranker
for name, model in rankers.items():
    scored = cands_df.copy()
    scored["p"] = model.predict_proba(X_e2e)[:, 1]
    topk = {c: g.sort_values("p", ascending=False)["product_id"].to_numpy()[:10]
            for c, g in scored.groupby("customer_unique_id")}
    results.append(e2e_metrics(topk, f"two_stage_{name}"))

e2e_table = pd.DataFrame(results).set_index("pipeline")
print(e2e_table.round(4).to_string())

# paired per-customer difference: two-stage XGBoost vs stage-1 order
d = e2e_hits["two_stage_xgboost"] - e2e_hits["stage1_only (routed hybrid/popularity)"]
lo_d, hi_d = bootstrap_ci(d)
print(f"\npaired diff two_stage_xgboost - stage1_only, hit@10: "
      f"{d.mean():+.4f} [{lo_d:+.4f}, {hi_d:+.4f}]")
import re
with mlflow.start_run(run_name="end_to_end"):
    mlflow.log_params({"stage": "end_to_end", "hybrid_w": W_HYBRID, "svd_k": K_SVD})
    for _, r in e2e_table.reset_index().iterrows():
        safe = re.sub(r"[^0-9a-zA-Z_.-]+", "_", r["pipeline"])[:40]
        mlflow.log_metric(f"hit10_{safe}", r["hit@10"])

                                        hit@10  hit@10_lo  hit@10_hi  ndcg@10  hit@10_warm  warm_lo  warm_hi  n_warm
pipeline                                                                                                            
stage1_only (routed hybrid/popularity)  0.0122     0.0105     0.0139   0.0062       0.0578   0.0325   0.0866     277
popularity_only                         0.0114     0.0099     0.0130   0.0052       0.0108   0.0000   0.0253     277
two_stage_logistic_regression           0.0061     0.0051     0.0073   0.0032       0.0144   0.0036   0.0289     277
two_stage_random_forest                 0.0132     0.0115     0.0150   0.0064       0.0433   0.0217   0.0686     277
two_stage_xgboost                       0.0146     0.0128     0.0165   0.0069       0.0325   0.0144   0.0542     277

paired diff two_stage_xgboost - stage1_only, hit@10: +0.0024 [+0.0008, +0.0041]


In [13]:
# Persist artifacts + metrics (reproducibility: saved models/configs)
import joblib
from scipy import sparse as sp

MODELS_DIR.mkdir(exist_ok=True)
sp.save_npz(MODELS_DIR / "item_item_similarity_fw.npz", cf_fw.S.astype(np.float32))
np.save(MODELS_DIR / "content_matrix_fw.npy", content_fw.P)
# persist the FEATURE-WINDOW SVD refit (the LLOO-build factors are a
# comparison-only artifact and would be leaky next to serving models)
svd_fw = SVDRecommender(n_components=K_SVD).fit(im_fw)
np.save(MODELS_DIR / "svd_item_factors_fw.npy", svd_fw.V)
(MODELS_DIR / "svd_item_factors.npy").unlink(missing_ok=True)
np.save(MODELS_DIR / "popularity_fw.npy", pop_fw.scores_)
pd.Series(im_fw.product_ids).to_csv(MODELS_DIR / "product_index.csv", index=False)
for name, model in rankers.items():
    path = MODELS_DIR / f"ranker_{name}.joblib"
    joblib.dump(model, path, compress=3)
    if path.stat().st_size > 90e6:  # GitHub hard limit is 100 MB
        path.unlink()
        print(f"{name}: artefact exceeds repo size budget - metrics kept, binary not persisted")

chosen = {
    "hybrid_w": float(W_HYBRID), "svd_k": int(K_SVD),
    "n_candidates": CFG["candidate_generation"]["n_candidates"],
    "negative_sampling": CFG["negative_sampling"],
    "windows": {k: [str(v[0].date()), str(v[1].date())] for k, v in windows.items()},
}
(MODELS_DIR / "chosen_config.yaml").write_text(yaml.safe_dump(chosen))
metrics_out = {
    "candidate_generation": cand_table.round(6).to_dict(orient="index"),
    "category_hit10": {k: float(v) for k, v in cat_rows.items()},
    "rankers": ranker_table.round(6).to_dict(orient="index"),
    "end_to_end": e2e_table.round(6).to_dict(orient="index"),
}
(MODELS_DIR / "metrics.json").write_text(json.dumps(metrics_out, indent=2))
sizes = {p.name: f"{p.stat().st_size / 1e6:.1f} MB" for p in MODELS_DIR.iterdir()
         if p.suffix in (".npz", ".npy", ".joblib")}
print(json.dumps(sizes, indent=2))

{
  "content_matrix_fw.npy": "10.7 MB",
  "item_item_similarity_fw.npz": "0.0 MB",
  "popularity_fw.npy": "0.1 MB",
  "ranker_logistic_regression.joblib": "0.0 MB",
  "ranker_random_forest.joblib": "53.2 MB",
  "ranker_xgboost.joblib": "0.6 MB",
  "svd_item_factors_fw.npy": "33.7 MB"
}


## Takeaways

- **Hybrid (w=0.25) wins candidate generation, and the paired test makes it
  decisive:** HitRate@10 0.203 vs content 0.171, with a paired per-user
  difference of +0.032 [+0.022, +0.042]. The independent CIs merely grazed;
  comparing on the same users settles it. CF (0.055), SVD (0.054) and
  popularity (0.031) are far behind. The hybrid weight itself is a coarse
  choice - the val spread across w is 0.0036 on 840 users, within noise.
- **Repurchase is the dominant signal for repeat buyers:** 69% of content's
  top-10 hits and 62% of hybrid's are products the customer already bought.
  The seen-item policy allows this deliberately and the table reports it.
- **Item-item CF is signal-starved on Olist:** 8,710 nonzero similarity pairs
  (8e-06 of the matrix) because delivered orders average 1.04 distinct
  products. The ranker's cf_signal feature is honest but nearly dead -
  positive for 0.1% of training positives (see the sanity plot).
- **SVD improves with rank (k=256 best at 0.054, at the top of the swept
  grid) but stays far below content/hybrid at every rank tried.** Chasing
  larger ranks isn't worth it for a model that isn't a contender.
- **Rankers:** XGBoost and random forest are effectively tied at ~0.865
  holdout AUC (LR 0.785). Thresholds come from out-of-fold predictions
  (training-fit probabilities are optimistically separated for the trees);
  F1 at tau: XGB 0.578, RF 0.569, LR 0.475. RF's 0.945 train AUC flags an
  overfitting gap for Step 5.
- **End-to-end: the two-stage XGBoost pipeline beats stage-1 order by a small
  but real margin** - paired per-customer diff +0.0024 [+0.0008, +0.0041],
  i.e. +20% relative on an overall HitRate@10 of 0.0146. LR re-ranking
  actively harms (0.0061). On the 277 warm customers the raw hybrid order
  looks better (0.0578 [0.033, 0.087] vs XGB 0.0325 [0.014, 0.054]) but the
  intervals overlap heavily - directional, underpowered, and left as future
  work (route-conditional re-ranking) rather than adopted post-hoc.
- **Routing shares:** 98.5% of holdout customers served by regional
  popularity, 1.5% by the hybrid - the cold-start reality made operational.
- **Final pipeline: hybrid candidates + XGBoost re-rank.**